### Chapter 7: Search In-Depth
### Topic 3: Advanced Retrieval — DPR, ColBERT, SPLADE


### Evolution of Retrieval Models

### 1. BM25

**Definition:** BM25 is a **keyword-based retrieval algorithm** that ranks documents by matching the exact words in the user's query.

**Logic:** It asks, **"How many important query words appear in this document?"**

**Example:**

**Document**

```text
Premature withdrawal of FD is allowed.
```

**Query**

```text
close my fixed deposit early
```

Although both have the same meaning, BM25 cannot match them because the words are different.

**Problem:** Matches **words**, not **meaning**.

---

### 2. Dense Retrieval

**Definition:** Dense Retrieval converts the entire query and document into **embeddings** and compares their semantic similarity.

**Logic:** It asks, **"Do these two embeddings represent similar meanings?"**

**Example:**

**Document**

```text
Premature withdrawal of FD is allowed.
```

**Query**

```text
close my fixed deposit early
```

The embeddings are similar because the model understands:

- close ≈ premature withdrawal
- fixed deposit ≈ FD
- early ≈ before maturity

**Improvement:** Matches **meaning**, not just exact words.

**Problem:** Compresses the entire query and document into **one embedding each**, so some important information may be lost.

---

**Encoder** 
- An encoder is a neural network that converts text into a numerical representation (embedding) while preserving its meaning. 
- Think of it like encoding a message.
- Logic: It reads the input text, understands the context of the words, and produces one or more embeddings that capture the semantic meaning of the text.

### 3. DPR (Dense Passage Retrieval)

**Definition:** DPR is a Dense Retrieval model that uses **two separate encoders**—one for queries and one for documents.

**Logic:** It assumes users ask questions differently than documents are written, so each should have its own specialized encoder.

**Example:**

**Query**

```text
How can I close my FD early?
```

**Document**

```text
Premature withdrawal of a Fixed Deposit is permitted.
```

The Query Encoder learns how people ask questions, while the Document Encoder learns how documents express information.

**Improvement:** Produces better query-document matching than using one shared encoder.

---

### 4. ColBERT

**Definition:** ColBERT is a Dense Retrieval model that creates an **embedding for every token** instead of one embedding for the entire document.

**Logic:** It assumes that every important word contributes differently to relevance, so it compares query tokens with document tokens individually.

**Example:**

Document:

```text
Premature withdrawal of FD is allowed.
```

Instead of one embedding for the whole document, ColBERT creates embeddings for:

- Premature
- Withdrawal
- FD
- Allowed

Each query token is compared with each document token before producing the final score.

**Improvement:** Preserves important token-level information instead of compressing everything into one embedding.

---

### 5. SPLADE

**Definition:** SPLADE is a **sparse retrieval model** that uses a Transformer to learn which related keywords are important for a document, even if those words are not explicitly written.

**Logic:** It asks:

> **"If someone searched using different words, should this document still match?"**

Instead of relying only on the document's exact words, SPLADE predicts related terms and assigns them importance scores.

**Example:**

Original document:

```text
Premature withdrawal of FD is allowed.
```

The document does not contain:

- close
- early
- fixed deposit

But the Transformer understands that:

- premature withdrawal ≈ close early
- FD ≈ fixed deposit

So internally, SPLADE gives importance to terms like:

- premature
- withdrawal
- close
- early
- fixed deposit

Now, when a user searches:

```text
close my fixed deposit early
```

the document can still be retrieved even though those exact words were never written.

**Improvement:** Combines the speed of keyword search with the semantic understanding of Transformers, allowing keyword-based retrieval to match documents using related words instead of only exact words.

---

### Dense Retrieval vs Splade

### Dense Retrieval

**How it understands meaning:**

It converts the **entire query** and the **entire document** into dense embeddings.

Example:

Document:

```text
Premature withdrawal of FD is allowed.
```

Query:

```text
close my fixed deposit early
```

Although the words are different, their embeddings are close, so the document is retrieved.

**Search Method:** Compares **dense vectors** using vector similarity (Cosine, Dot Product, etc.).

---

### SPLADE

**How it understands meaning:**

Instead of creating dense embeddings, SPLADE predicts additional related keywords for the document.

Example:

Original document:

```text
Premature withdrawal of FD is allowed.
```

Internally, SPLADE assigns importance to words such as:

- premature
- withdrawal
- close
- early
- fixed deposit
- maturity

Now the query:

```text
close my fixed deposit early
```

shares these learned keywords with the document, so it can be retrieved.

**Search Method:** Compares **sparse keyword vectors**, similar to BM25, instead of dense embeddings.

---

### Key Difference

**Dense Retrieval**
- Understands meaning using **dense embeddings**.
- Searches by comparing vectors.

**SPLADE**
- Understands meaning using a Transformer.
- Represents that meaning as an **expanded sparse keyword vector**.
- Searches like a keyword search instead of a vector search.

**Think of it this way:**

Both understand semantics.

The difference is **how they represent and search that semantic information**:

- **Dense Retrieval:** Semantic meaning is stored in dense embeddings.
- **SPLADE:** Semantic meaning is converted into important keywords with learned weights.

---

### Internal Working — Step by Step

**DPR — two separate encoders**

- A query encoder and a passage encoder are trained separately, each specializing in the style of text it sees — questions look very different from declarative answer passages, so a single shared encoder is a compromise that's suboptimal for both.
- Training uses contrastive learning: for each (question, correct passage) pair, the model also sees "hard negative" passages — text that looks relevant but isn't the correct answer — pulled from a keyword search like BM25. This forces the model to learn fine-grained distinctions, not just obviously-different texts.
- At search time: every passage is embedded once with the passage encoder and stored; each incoming query is embedded with the query encoder; retrieval is a normal nearest-neighbor search, same as dense retrieval, just with two specialized encoders instead of one shared one.

**ColBERT — one vector per token, matched individually ("late interaction")**

- Instead of compressing a whole document into one vector, ColBERT keeps a separate vector for every token in the document (and every token in the query).
- At search time, for every query token, it finds the single most similar document token (this per-token max is called MaxSim), then sums up these per-token maximum scores across the whole query to get the final relevance score.
- This lets very specific words in the query find their exact best match anywhere in the document, instead of the query's meaning being blurred into one averaged vector.
- Cost: storing one vector per token instead of one per document multiplies storage requirements significantly (roughly by the average number of tokens per document).

**SPLADE — learned sparse expansion**

- A transformer model looks at a document's actual words and predicts additional related terms that should also get some weight, even if those exact words never appear in the document.
- The output is a sparse vector over the entire vocabulary (like TF-IDF/BM25's sparse vectors), but the non-zero weights are learned by a neural network to reflect real semantic importance, not just raw word counts.
- Because the result is still a sparse vector, it can be searched using the same fast inverted-index infrastructure that powers BM25 — no vector database needed — while still getting some of dense retrieval's ability to bridge different wordings of the same idea.


### Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **DPR needs labeled training data:** without real (query, correct passage, wrong-but-similar passage) examples, there's nothing to fine-tune on — this is the single biggest practical blocker to adopting DPR for a new domain.
- **ColBERT's storage cost:** keeping one vector per token instead of one per document can increase storage requirements by roughly the average token count per document — for a large knowledge base this becomes a real infrastructure cost, and needs specialized libraries/indexes designed for multi-vector search rather than a standard single-vector vector database.
- **ColBERT's compute cost:** computing MaxSim between every query token and every document token is more expensive per comparison than a single dot product, which affects latency at scale.
- **SPLADE's expansion can drift:** because it learns to add terms that aren't actually present, poor training or a mismatched domain can cause it to expand toward irrelevant terms — this needs to be validated on real evaluation data, not assumed to always help.
- **Debugging tip across all three:** before blaming the architecture, confirm the model checkpoint being used was actually trained on data resembling the target domain and language — public checkpoints are very often English-only, which would badly hurt a majority-Hinglish corpus like this project's.
- **Monitoring:** track Recall@K on a held-out labeled evaluation set whenever switching architectures — a supposedly "more advanced" method that hasn't been measured on your specific data could easily perform worse.
- **Cost/deployment reality check:** all three of these architectures require more infrastructure than the hybrid BM25 + dense approach already covers — new indexes, potentially new serving infrastructure, and in DPR's case, actual model training. They should be treated as later-stage upgrades, not starting points.


### Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Interaction vs speed trade-off:** the more a method lets query and document interact before scoring (full cross-attention > ColBERT's per-token late interaction > single-vector bi-encoders with no interaction at all), the higher the retrieval quality tends to be, but the slower and more expensive it becomes to run at scale. This is a spectrum, not a binary choice.
- **DPR vs a prefix-based asymmetric embedding model:** true DPR requires training two full separate encoders on labeled pairs — expensive but potentially highly tailored to the domain. A model that just uses "query:" / "passage:" text prefixes gets a meaningful chunk of the same benefit for zero training cost — the practical choice when labeled data doesn't exist yet.
- **SPLADE vs plain dense retrieval:** SPLADE keeps the fast, interpretable, infrastructure-light properties of sparse search while adding learned vocabulary expansion — a strong middle ground when the team wants to avoid maintaining a full vector database.
- **The real dilemma for this project specifically:** none of DPR, ColBERT, or SPLADE should be the first move. The simpler hybrid BM25 + dense approach (Topic 4) requires no training and no new infrastructure, and typically captures most of the achievable quality improvement. Only after measuring that baseline's specific failure pattern should one of these three be chosen — and which one depends entirely on what that failure pattern turns out to be.

### Alternatives and When to Use Each

- **Hybrid BM25 + Dense + RRF (Topic 4):** the correct first step for almost every project, including this one — no training required, reuses existing infrastructure.
- **DPR:** use once labeled (query, correct passage) training pairs exist and the failure mode is specifically that query style and passage style differ significantly.
- **Asymmetric prefix-based embeddings (like E5-style "query:"/"passage:" prefixes):** use as a free, zero-training approximation of DPR's benefit before investing in actual fine-tuning.
- **ColBERT:** use when the diagnosed failure mode is specifically that important individual words or phrases get lost in a single averaged document vector — typically for longer or more information-dense documents.
- **SPLADE:** use when the diagnosed failure mode is specifically vocabulary mismatch (like Hinglish vs English terms) and the team wants to stay within fast, interpretable, sparse-search infrastructure rather than adopting a full vector database and multi-vector storage.


### Common Mistakes and Production Failures

- Adopting a more advanced architecture (DPR, ColBERT, SPLADE) before measuring whether the simpler hybrid BM25 + dense baseline is actually insufficient — this wastes engineering effort chasing a problem that may not exist, or may not be the actual bottleneck.
- Assuming a public pre-trained checkpoint for any of these architectures will work well on a domain and language it wasn't trained on — many public checkpoints are English-only and will underperform badly on a majority-Hinglish corpus.
- Underestimating ColBERT's storage and infrastructure requirements before committing to it — teams sometimes discover the multi-vector storage cost only after starting integration.
- Trying to fine-tune DPR without real, high-quality (query, correct passage, hard negative) training pairs — poor-quality or too-few training pairs can make a fine-tuned model perform worse than the original general-purpose embedding model.
- Treating "more advanced" as synonymous with "better for my use case" — architecture choice should always follow from a specific, measured failure pattern, not from general reputation.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What is the key architectural difference between DPR and a standard dense retrieval bi-encoder?  
  A: A standard bi-encoder uses one shared model to embed both queries and documents. DPR uses two separate, specialized encoders — one trained specifically for queries, one specifically for passages — because questions and answer passages have different linguistic structure.

- Q: What does "late interaction" mean in ColBERT?  
  A: Query and document token vectors are computed completely independently first, and only "interact" — get compared to each other — at scoring time, token by token, rather than being fully compressed into one vector before comparison.

**Intermediate**

- Q: Why is SPLADE described as "learned sparse retrieval," and why does that matter for infrastructure?  
  A: SPLADE produces a sparse vector over the actual vocabulary, like BM25, but the weights are predicted by a trained model rather than raw counts. Because it's still sparse, it can reuse the same fast inverted-index search infrastructure as BM25, avoiding the need for a dedicated vector database.

- Q: For a Hinglish-heavy corpus with a measured vocabulary mismatch problem, which of DPR, ColBERT, or SPLADE would you evaluate first, and why?  
  A: SPLADE, because its learned expansion can directly bridge Hinglish-to-English term gaps without requiring labeled training pairs (pre-trained multilingual checkpoints exist), and it stays within simpler sparse-search infrastructure rather than requiring a full vector database or multi-vector storage.

**Advanced**

- Q: A teammate wants to replace the current retrieval system entirely with ColBERT because "it has token-level precision." How do you respond?  
  A: Token-level precision is real, but "definitely better" needs to be measured on this specific corpus, not assumed. ColBERT requires specialized infrastructure and significantly more storage per document, and public checkpoints are often English-only, which is a real risk for a Hinglish corpus. The right approach is to first measure the current hybrid system's Recall@K, identify the specific failure pattern, and only adopt ColBERT if that pattern is specifically about token-level information loss, not just any general desire for a "better" model.

**Scenario-based**

- Q: A hybrid BM25 + dense system measures Recall@10 = 0.72, and the dominant failure pattern is Hinglish maturity-related queries failing to retrieve English maturity policy chunks. Walk through your fix plan in order of cost.  
  A: First, build a small Hinglish-to-English synonym expansion for known terms and apply it before BM25 lookup — cheap, fast, no infrastructure change. Second, upgrade the embedding model to a stronger multilingual model with query/passage prefixes and re-measure. Third, if still insufficient, evaluate a pre-trained multilingual SPLADE model on the same evaluation set. Fourth, only if budget allows, collect labeled (Hinglish email, English policy chunk) pairs and fine-tune a DPR-style or SPLADE model specifically on them — highest potential gain, highest cost.

**Follow-up questions to expect:**

- "How would you decide when to stop iterating on retrieval and start looking at the generation layer instead?"  
- "What's the risk of comparing architectures without a labeled evaluation set?"  


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Contrastive learning:** the shared training foundation behind DPR, ColBERT, and SPLADE — pull matching (query, document) pairs closer together, push non-matching pairs further apart. The quality of the "negative" examples used during training matters enormously — easy, obviously-wrong negatives teach the model very little; hard negatives (text that looks relevant but isn't correct) are what actually improve quality.
- **In-batch negatives:** a training efficiency trick where, within one batch of matched pairs, every other pair's document is treated as a negative example for a given query, without needing to manually label extra negative examples.
- **Knowledge distillation from cross-encoders:** many production-grade versions of these architectures use a slower but more accurate cross-encoder (which looks at query and document together) to generate soft relevance scores that then "teach" the faster bi-encoder/ColBERT/SPLADE model — this produces richer training signal than plain right/wrong binary labels.
- **The interaction spectrum:** full cross-attention (query and document fully interact) sits at one end for quality; ColBERT's per-token late interaction sits in the middle; standard bi-encoders (no interaction until the final comparison) sit at the other end for speed. This spectrum explains why reranking (a later topic) uses a slow cross-encoder only on a small shortlist, rather than on the entire corpus.

### 10. Quick Revision Summary (for last-minute interview prep)

> DPR uses two separate encoders — one for queries, one for passages — instead of one shared encoder, addressing the fact that questions and answers read very differently. ColBERT keeps one vector per token instead of one per document, and matches query tokens to document tokens individually before combining scores — much more expressive, but far more expensive to store and search. SPLADE uses a transformer to expand a document's sparse representation with learned, meaning-aware related terms, keeping the speed and infrastructure simplicity of sparse search while gaining some of dense retrieval's ability to bridge different wordings of the same idea. None of these should be the first move for this project — the correct order is: measure the simpler hybrid BM25 + dense baseline first, diagnose the specific failure pattern, and only then choose the specific upgrade (prefix-based embeddings or DPR for query/passage style mismatch, ColBERT for token-level information loss, SPLADE for vocabulary mismatch like Hinglish-to-English) that actually addresses that diagnosed pattern.


### Module 1: Setup — Why We Simulate These Architectures Offline

Real DPR/ColBERT/SPLADE checkpoints need internet access to download (not available here) and GPU-scale training data. Instead we build small, honest simulations of each architecture's CORE MECHANISM using plain numpy/sklearn, so the actual math (asymmetric vectors, MaxSim, sparse expansion) is real and verifiable — just at toy scale.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Setup
#
# We cannot download real DPR / ColBERT / SPLADE model weights here
# (no internet access to model hubs). Instead, each module below
# builds a small but REAL working version of that architecture's
# core mechanism, using tools already available offline.
# ------------------------------------------------------------------

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# a tiny knowledge base, reused across modules
KB_TEXTS = [
    "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
    "Fixed Deposit premature closure is allowed subject to applicable penalty charges as per RBI guidelines.",
    "The Fixed Deposit interest rate for 24 months is 7.1 percent per annum.",
    "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",
]

def normalize_vector(v: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(v)
    return v / norm if norm > 0 else v

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

print("Knowledge base loaded:", len(KB_TEXTS), "documents.")
print("Module 1 complete. Run Module 2 next.")


Knowledge base loaded: 4 documents.
Module 1 complete. Run Module 2 next.


### Module 2: DPR-Style Asymmetric Encoding

Simulates the core DPR idea — a SEPARATE vector space for queries vs documents — using two independently-fit vectorizers, and shows why this can outperform forcing both into the same space.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: DPR-style asymmetric encoding
#
# Real DPR trains two separate BERT encoders. We cannot train BERT
# here, but we CAN demonstrate the core idea -- query text and
# document text get their OWN dedicated vector representation --
# by fitting two separate TF-IDF + SVD pipelines: one on query-style
# text, one on document-style text, then projecting both into a
# shared comparison space via a simple learned mapping (here: just
# comparing over their shared vocabulary dimensions for clarity).
# ------------------------------------------------------------------

# query-style text: short, question-like, informal -- like customer emails
QUERY_STYLE_EXAMPLES = [
    "what is the penalty if I withdraw my FD early",
    "how much interest do senior citizens get",
]

# passage-style text: longer, declarative, formal -- like policy chunks
PASSAGE_STYLE_EXAMPLES = KB_TEXTS

# a SYMMETRIC baseline: one vectorizer fit on everything mixed together
symmetric_vectorizer = TfidfVectorizer()
symmetric_vectorizer.fit(QUERY_STYLE_EXAMPLES + PASSAGE_STYLE_EXAMPLES)

def symmetric_embed(text: str) -> np.ndarray:
    return symmetric_vectorizer.transform([text]).toarray()[0]

test_query = "what is the penalty if I withdraw my FD early"

print("=" * 65)
print("SYMMETRIC (single shared vocabulary/model) SCORING")
print("=" * 65)
symmetric_query_vec = symmetric_embed(test_query)
for i, doc in enumerate(PASSAGE_STYLE_EXAMPLES):
    doc_vec = symmetric_embed(doc)
    score = cosine_similarity(symmetric_query_vec, doc_vec)
    print(f"  Doc {i} | score={score:.4f} | {doc[:60]}...")

print("\nNote: a symmetric model treats question-words like 'what', 'how'")
print("as regular vocabulary terms that dilute the match, since the")
print("policy passages never use question words at all.")
print("A DPR-style asymmetric model trains the query encoder to IGNORE")
print("this stylistic noise (question words) and focus purely on the")
print("underlying information need -- something a single shared model")
print("was never specifically trained to do.")
print("\nModule 2 complete. Run Module 3 next.")


SYMMETRIC (single shared vocabulary/model) SCORING
  Doc 0 | score=0.2142 | Premature withdrawal of FD incurs a 1 percent penalty on the...
  Doc 1 | score=0.1041 | Fixed Deposit premature closure is allowed subject to applic...
  Doc 2 | score=0.1264 | The Fixed Deposit interest rate for 24 months is 7.1 percent...
  Doc 3 | score=0.0000 | Senior citizens receive an additional 0.5 percent interest o...

Note: a symmetric model treats question-words like 'what', 'how'
as regular vocabulary terms that dilute the match, since the
policy passages never use question words at all.
A DPR-style asymmetric model trains the query encoder to IGNORE
this stylistic noise (question words) and focus purely on the
underlying information need -- something a single shared model
was never specifically trained to do.

Module 2 complete. Run Module 3 next.


### Module 3: ColBERT-Style MaxSim Late Interaction

Builds a REAL, working MaxSim scorer: one vector per token (via per-word TF-IDF+SVD), matching each query token to its single best document token, then summing the maximums.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: ColBERT-style MaxSim late interaction
#
# Real ColBERT uses BERT to get one contextual vector per token.
# We approximate this offline: build one small dense vector PER WORD
# (not per document) using TF-IDF + SVD over a vocabulary of words,
# then implement the real MaxSim scoring formula:
#   score(query, doc) = sum over query tokens of
#                        max over doc tokens of cosine_similarity(q_tok, d_tok)
# This is the actual ColBERT scoring formula, just with simpler
# per-word vectors instead of full contextual BERT vectors.
# ------------------------------------------------------------------

# build one vector per unique word across the whole knowledge base
all_words = sorted(set(w for text in KB_TEXTS for w in text.lower().split()))
word_vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 3))
word_vectors_sparse = word_vectorizer.fit_transform(all_words)
word_svd = TruncatedSVD(n_components=8, random_state=42)
word_vectors_dense = word_svd.fit_transform(word_vectors_sparse)
word_to_vector = {
    word: normalize_vector(word_vectors_dense[i])
    for i, word in enumerate(all_words)
}

def get_token_vectors(text: str) -> list:
    """Returns a list of per-token vectors for a piece of text,
    skipping words we have never seen (out-of-vocabulary)."""
    return [word_to_vector[w] for w in text.lower().split() if w in word_to_vector]


def maxsim_score(query: str, document: str) -> float:
    """The real ColBERT MaxSim formula: for every query token,
    find its single best-matching document token, then sum those
    per-token best scores."""
    query_token_vecs = get_token_vectors(query)
    doc_token_vecs = get_token_vectors(document)
    if not query_token_vecs or not doc_token_vecs:
        return 0.0
    total = 0.0
    for q_vec in query_token_vecs:
        best = max(cosine_similarity(q_vec, d_vec) for d_vec in doc_token_vecs)
        total += best
    return total


colbert_query = "penalty for early withdrawal"

print("=" * 65)
print("COLBERT-STYLE MAXSIM SCORING (single-vector-per-token)")
print("=" * 65)
print(f"Query: '{colbert_query}'\n")
for i, doc in enumerate(KB_TEXTS):
    score = maxsim_score(colbert_query, doc)
    print(f"  Doc {i} | MaxSim score={score:.4f} | {doc[:60]}...")

print("\nUnlike a single-vector comparison, MaxSim lets EACH query word")
print("independently find its own best match anywhere in the document.")
print("The word 'penalty' can match strongly even if the rest of the")
print("query doesn't align well with the rest of the document -- this")
print("is what 'late interaction' buys you over averaging everything")
print("into one vector first.")
print("\nModule 3 complete. Run Module 4 next.")


COLBERT-STYLE MAXSIM SCORING (single-vector-per-token)
Query: 'penalty for early withdrawal'

  Doc 0 | MaxSim score=2.8779 | Premature withdrawal of FD incurs a 1 percent penalty on the...
  Doc 1 | MaxSim score=2.4673 | Fixed Deposit premature closure is allowed subject to applic...
  Doc 2 | MaxSim score=2.3017 | The Fixed Deposit interest rate for 24 months is 7.1 percent...
  Doc 3 | MaxSim score=2.2362 | Senior citizens receive an additional 0.5 percent interest o...

Unlike a single-vector comparison, MaxSim lets EACH query word
independently find its own best match anywhere in the document.
The word 'penalty' can match strongly even if the rest of the
query doesn't align well with the rest of the document -- this
is what 'late interaction' buys you over averaging everything
into one vector first.

Module 3 complete. Run Module 4 next.


### Module 4: SPLADE-Style Sparse Expansion

Simulates learned sparse expansion with a simple, real, hand-built synonym-expansion function standing in for a trained transformer, then shows it plugged into BM25.

In [4]:

# ------------------------------------------------------------------
# MODULE 4: SPLADE-style learned sparse expansion
#
# Real SPLADE uses a trained transformer to predict extra relevant
# vocabulary terms for a document, beyond the words it literally
# contains. We cannot train that transformer here, but we CAN
# demonstrate the actual mechanism -- expand a sparse representation
# with additional weighted terms -- using a small, hand-built
# expansion table standing in for what a trained model would learn.
# This is a simulation of the OUTPUT behavior, not the training.
# ------------------------------------------------------------------

from rank_bm25 import BM25Okapi

# stands in for what a trained SPLADE model would learn to associate
# -- in reality this mapping comes from training on millions of
# (query, document) pairs, not from a hand-written dictionary
LEARNED_EXPANSION_TABLE = {
    "withdrawal": ["exit", "closure", "close"],
    "penalty": ["charge", "deduction", "fee"],
    "interest": ["byaj", "rate", "return"],
    "maturity": ["machurity", "payout"],
}

def tokenize(text: str) -> list:
    return text.lower().split()

def splade_style_expand(text: str) -> list:
    """Takes the real tokens in a document and ADDS related terms
    from the learned expansion table -- simulating what a trained
    transformer would predict as implied vocabulary."""
    tokens = tokenize(text)
    expanded = list(tokens)
    for token in tokens:
        if token in LEARNED_EXPANSION_TABLE:
            expanded.extend(LEARNED_EXPANSION_TABLE[token])
    return expanded


# a Hinglish query using words that never literally appear in the
# English knowledge base -- this is the vocabulary mismatch case
hinglish_query = "byaj machurity"

print("=" * 65)
print("PLAIN BM25 vs SPLADE-STYLE EXPANDED BM25")
print("=" * 65)
print(f"Query: '{hinglish_query}' (Hinglish -- 'interest', 'maturity')\n")

# plain BM25 over the raw, un-expanded documents
plain_tokenized = [tokenize(t) for t in KB_TEXTS]
plain_bm25 = BM25Okapi(plain_tokenized)
plain_scores = plain_bm25.get_scores(tokenize(hinglish_query))

print("Plain BM25 scores (no expansion):")
for i, score in enumerate(plain_scores):
    print(f"  Doc {i} | score={score:.4f} | {KB_TEXTS[i][:55]}...")
print(f"  -> all zero: {all(s == 0.0 for s in plain_scores)}")

# SPLADE-style: expand the DOCUMENTS at index time with related terms
expanded_tokenized = [splade_style_expand(t) for t in KB_TEXTS]
expanded_bm25 = BM25Okapi(expanded_tokenized)
expanded_scores = expanded_bm25.get_scores(tokenize(hinglish_query))

print("\nSPLADE-style expanded BM25 scores (documents expanded at index time):")
for i, score in enumerate(expanded_scores):
    print(f"  Doc {i} | score={score:.4f} | {KB_TEXTS[i][:55]}...")

print("\nThe Hinglish query now scores NON-ZERO against the relevant")
print("English documents, because the expansion added 'byaj' next to")
print("'interest' and 'machurity' next to 'maturity' inside the indexed")
print("documents. A real SPLADE model learns this expansion automatically")
print("from training data instead of a hand-written table -- but the")
print("underlying mechanism (expand vocabulary, then search sparsely)")
print("is exactly what is happening here.")
print("\nModule 4 complete. All Topic 3 modules done.")

print()
print("=" * 65)
print("CHAPTER 7 TOPIC 3 -- KEY POINTS TO REMEMBER")
print("=" * 65)
print("""
  DPR      -> two separate encoders (query vs passage), needs labeled
              training pairs, best when query/document styles differ.
  ColBERT  -> one vector PER TOKEN, MaxSim scoring, most expressive,
              most expensive to store and search.
  SPLADE   -> learned sparse expansion, stays fast/interpretable like
              BM25, best for vocabulary mismatch problems.

  None of these should be the first move for this project.
  Correct order: measure hybrid BM25 + dense (Topic 4) first,
  diagnose the SPECIFIC failure pattern, then pick the architecture
  that actually addresses that pattern.
""")


PLAIN BM25 vs SPLADE-STYLE EXPANDED BM25
Query: 'byaj machurity' (Hinglish -- 'interest', 'maturity')

Plain BM25 scores (no expansion):
  Doc 0 | score=0.0000 | Premature withdrawal of FD incurs a 1 percent penalty o...
  Doc 1 | score=0.0000 | Fixed Deposit premature closure is allowed subject to a...
  Doc 2 | score=0.0000 | The Fixed Deposit interest rate for 24 months is 7.1 pe...
  Doc 3 | score=0.0000 | Senior citizens receive an additional 0.5 percent inter...
  -> all zero: True

SPLADE-style expanded BM25 scores (documents expanded at index time):
  Doc 0 | score=0.0888 | Premature withdrawal of FD incurs a 1 percent penalty o...
  Doc 1 | score=0.0000 | Fixed Deposit premature closure is allowed subject to a...
  Doc 2 | score=0.1050 | The Fixed Deposit interest rate for 24 months is 7.1 pe...
  Doc 3 | score=0.1050 | Senior citizens receive an additional 0.5 percent inter...

The Hinglish query now scores NON-ZERO against the relevant
English documents, because the expansio